# Derivation: Exponentially Weighted GBM Parameter Estimates
The [L5a lecture](../../CHEME-5660-L5a-Lecture-SAGBM-NPV-Fall-2026.ipynb) states the EMA update equations. Here, we develop the algebra behind that result, using the same notation and estimates as the [worked EMA example](../../CHEME-5660-L5a-Example-EMA-SAGBM-Fall-2026.ipynb).

> __Learning Objectives:__
>
> By the end of this derivation, you should be able to:
>
> * **Derive the exponential weights:** Expand the recursive mean update and relate the decay factor to an observation's half-life.
> * **Derive the centered variance update:** Use the weighted second moment to account for the changing mean when updating variance.
> * **Recover GBM parameter estimates:** Convert growth-rate moments to mean growth, volatility, and arithmetic drift, and explain how to use them within a forecast window.

The out-of-sample model uses one set of historical estimates throughout 2025. What happens if the recent observations suggest a different mean growth rate or volatility? An __exponential moving average (EMA)__ gives more weight to recent observations, allowing the estimates to change as new prices arrive.

Why update parameters if GBM assumes they are constant? We use GBM as a local forecast model. Its parameters remain fixed over each forecast window, then we update the estimates after another observation arrives. This produces a sequence of forecasts, each based on the information available when it is made. Updating can respond to observed changes; it cannot anticipate the next unobserved shock.

___


## Exponential Weights and Half-Life
Let $S_k>0$ be the observed price at row $k$ and let $\Delta t>0$ be the time between observations in years. The observed growth rate, in inverse years, is given by:

$$
g_k=\frac{1}{\Delta t}\ln\!\left(\frac{S_k}{S_{k-1}}\right).
$$

Under the fixed-parameter GBM model, the growth rate has mean $\mu_g$ and variance $\sigma^2/\Delta t$. We update these two moments directly.

Let $m_k$ and $v_k$ denote our estimated mean and variance of the observed growth rates after observing row $k$. At the selected entry row $s$, initialize them from the 2014–2024 estimates $\hat\mu_{g,0}$ and $\hat\sigma_0$:

$$
m_s=\hat\mu_{g,0},\qquad v_s=\frac{\hat\sigma_0^2}{\Delta t}.
$$

The training mean-growth estimate comes from log-price regression. We use it to initialize the mean growth; it need not equal the sample mean of the training growth rates. The first update uses the growth rate observed from row $s$ to row $s+1$.

Choose a decay factor $0<\lambda<1$. After each new observation, retain a fraction $\lambda$ of the previous mean and give the new growth rate a weight of $1-\lambda$:

$$
\begin{aligned}
m_k&=\lambda m_{k-1}+(1-\lambda)g_k\\
&=m_{k-1}+(1-\lambda)\underbrace{(g_k-m_{k-1})}_{\text{observed growth rate minus previous mean}},\qquad k>s.
\end{aligned}
$$

Thus, an observed growth rate above the previous mean raises the estimate, while one below it lowers the estimate.

Why are the weights exponential? Repeated substitution, after $n=k-s$ new observations, gives:

$$
m_{s+n}=\underbrace{\lambda^n m_s}_{\text{training baseline}}
+(1-\lambda)\sum_{j=1}^{n}\lambda^{n-j}g_{s+j}.
$$

The weights sum to one, including the weight on the baseline. Each new observation multiplies all previous weights by $\lambda$, so older observations gradually lose influence. The limiting choice $\lambda=1$ retains the frozen training estimates.

> __Half-life of an observation's weight:__ Let $H_{\mathrm{half}}>0$ be the number of trading observations over which a weight falls to one half of its current value. Choose the decay factor using:
>
> $$
> \lambda^{H_{\mathrm{half}}}=\frac12
> \quad\Longrightarrow\quad
> \lambda=2^{-1/H_{\mathrm{half}}}.
> $$
>
> With a half-life of 21 observations, $\lambda\approx0.9675$, and each new growth-rate observation receives weight $1-\lambda\approx0.0325$. A shorter half-life responds more quickly to recent changes but also follows more of their noise. The half-life controls estimation memory; it is separate from the forecast holding period.

___


## Centered Variance and GBM Parameters
We also need the spread of the growth rates around their changing mean. To derive its update, let $q_k$ denote the exponentially weighted mean of the squared growth rates. Initialize $q_s=v_s+m_s^2$ and use the same weights as for the mean:

$$
q_k=\lambda q_{k-1}+(1-\lambda)g_k^2,\qquad v_k=q_k-m_k^2.
$$

Substitute $q_{k-1}=v_{k-1}+m_{k-1}^2$ and the mean update, then collect terms:

$$
\begin{aligned}
v_k
&=\lambda\left(v_{k-1}+m_{k-1}^2\right)+(1-\lambda)g_k^2
-\left[\lambda m_{k-1}+(1-\lambda)g_k\right]^2\\
&=\lambda v_{k-1}+\lambda(1-\lambda)(g_k-m_{k-1})^2.
\end{aligned}
$$

The factor $\lambda(1-\lambda)$ accounts for the change in the weighted mean. Simply averaging squared deviations from the previous mean with weight $1-\lambda$ would give a different variance estimate.

> __Centered exponential updates:__ Define the new deviation as $\delta_k=g_k-m_{k-1}$. The mean and variance updates used in our example are:
>
> $$
> \boxed{\begin{aligned}
> m_k&=m_{k-1}+(1-\lambda)\delta_k, &&\text{(mean update)}\\
> v_k&=\lambda\left[v_{k-1}+(1-\lambda)\delta_k^2\right]. &&\text{(variance update)}
> \end{aligned}}
> $$
>
> These are weighted population moments, without a finite-sample unbiased correction. The variance remains nonnegative when initialized with $v_s\geq0$. We update growth-rate variance before converting it to GBM volatility.

Recall that $\mathbb E[g_k]=\mu_g$ and $\operatorname{Var}(g_k)=\sigma^2/\Delta t$. The estimated variance $v_k$ has units of $\mathrm{year}^{-2}$, so the GBM volatility estimate is $\sqrt{v_k\Delta t}$, with units of $\mathrm{year}^{-1/2}$. The parameter estimates are given by:

$$
\boxed{\begin{aligned}
\hat\mu_{g,k}&=m_k,\qquad
\hat\sigma_k=\sqrt{v_k\Delta t},\\
\hat\mu_k&=\hat\mu_{g,k}+\frac12\hat\sigma_k^2.
\end{aligned}}
$$

Mean growth $\hat\mu_{g,k}$ and arithmetic drift $\hat\mu_k$ have units of inverse years; volatility $\hat\sigma_k$ has units of inverse square-root years. The last equation preserves the distinction between mean growth and drift developed in L4b. Even when we hold mean growth fixed and update only volatility, the corresponding arithmetic drift changes through the $\hat\sigma_k^2/2$ term.

___


## Forecasts from the Updated Estimates
At forecast row $k$, the update uses only prices observed through that row. Let $H\geq1$ be the number of trading intervals in the forward window and let $h=H\Delta t$ be its duration in years. Holding the current estimates fixed, the local GBM model gives the following future-price distribution:

$$
S_{k+H}=S_k\exp\!\left[\hat\mu_{g,k}h+\hat\sigma_k\sqrt h\,Z\right],
\qquad Z\sim\mathcal N(0,1).
$$

We can use this distribution to simulate paths or calculate an NPV target probability. When the next price arrives, we update the estimates and issue a new forecast. We do not put subsequently observed parameter updates into an earlier forecast.

Giving recent observations more weight changes the forecasts, but does it improve them? The example compares frozen training estimates, updated volatility with frozen mean growth, and updated mean growth with volatility. Both updating methods use the same centered variance estimate; the volatility-only forecast holds its mean growth at the training value. The example evaluates their probabilities against the same later outcomes. This separates the effect of updating volatility from the effect of updating mean growth, which can be especially noisy over a short estimation window.

> [▶ Update GBM parameters with an exponential moving average](../../CHEME-5660-L5a-Example-EMA-SAGBM-Fall-2026.ipynb). Can giving recent observations more weight improve the forecasts? We initialize mean growth and volatility from the 2014–2024 estimates, update them as 2025 prices arrive, and compare their forecasts with the frozen model. We measure whether updating volatility alone or both parameters improves the coverage and width of the forecasts' 95% growth-rate prediction bands.

The example applies these updates while keeping the original purchase fixed. It recalculates the target probability each day for a sale at the end of an adjustable forward window, initially 21 trading intervals, and compares the resulting simulations and forecast scores.

The [local parameter-update helper](../../src/AdaptiveGBM.jl) implements these centered moment equations.

___


## Summary
We developed the exponentially weighted moments used to update the GBM parameter estimates as new prices arrive.

> __Key Takeaways:__
>
> * **Exponential weights:** We expanded the mean recursion to show how the training baseline and earlier observations lose weight. The half-life determines the rate of that decay.
> * **Centered variance:** We subtracted the squared weighted mean from the weighted second moment. This accounts for the changing mean and gives the factor of $\lambda(1-\lambda)$ multiplying the squared new deviation.
> * **GBM forecasts:** We converted the growth-rate moments to mean growth, volatility, and arithmetic drift. Each forecast holds its current estimates fixed; the worked example checks whether subsequent updates improve predictions.

___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.
